# Cogni AP Tutor — teacher labeling and compilation

Run cells from top to bottom on a CPU runtime. The labeling cell is resumable. After it
finishes, the final cells create `MyDrive/cogni/production_v1` and `production_v1.zip`.


In [ ]:
!pip -q install requests

from google.colab import drive
drive.mount("/content/drive")

import getpass
import json
import os
import random
import shutil
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

BUNDLE = Path("/content/drive/MyDrive/cogni/ap_teacher_labeling_bundle.zip")
WORK = Path("/content/cogni-labeling")

assert BUNDLE.exists(), f"Missing: {BUNDLE}"
if WORK.exists():
    shutil.rmtree(WORK)
shutil.unpack_archive(BUNDLE, WORK)

INPUT_FILE = (
    WORK
    / "datasets/training_candidates/ap_fallacy_production_v1/labeling_input.jsonl"
)
base_url = "https://tfy.promptlens.trilogy.com/v1"
model = "gemini-group/gemini-3.1-pro"
os.environ["TEACHER_API_KEY"] = getpass.getpass("Paste TEACHER_API_KEY: ").strip()
assert os.environ["TEACHER_API_KEY"]

all_inputs = [
    json.loads(line)
    for line in INPUT_FILE.read_text().splitlines()
    if line.strip()
]
print("Inputs:", len(all_inputs))
assert len(all_inputs) == 15_000


In [ ]:
import requests

BEHAVIOR_SCHEMA = {
    "type": "object",
    "properties": {
        "argument_summary": {"type": "string"},
        "assumptions": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 1,
        },
        "primary_fallacy": {"type": "string"},
        "why_this_applies": {"type": "string"},
        "cross_domain_analogy": {
            "type": "object",
            "properties": {
                "analogy": {"type": "string"},
                "mapping": {"type": "string"},
            },
            "required": ["analogy", "mapping"],
            "additionalProperties": False,
        },
        "transfer_check": {
            "type": "object",
            "properties": {
                "example": {"type": "string"},
                "question": {"type": "string"},
            },
            "required": ["example", "question"],
            "additionalProperties": False,
        },
        "reflective_question": {"type": "string"},
    },
    "required": [
        "argument_summary",
        "assumptions",
        "primary_fallacy",
        "why_this_applies",
        "cross_domain_analogy",
        "transfer_check",
        "reflective_question",
    ],
    "additionalProperties": False,
}


def build_prompt(example):
    return f'''You are an AP English Language logical-fallacy tutor.

Analyze the argument and return exactly one JSON object matching the supplied schema.

Requirements:
1. Summarize the argument.
2. Identify its assumptions.
3. Name exactly one primary fallacy, or say "No clear fallacy".
4. Explain why.
5. Give one analogy from a different domain and map it back.
6. Give one new transfer example and ask the student to classify it without revealing the answer.
7. End with exactly one reflective question.

ARGUMENT:
{example["essay"]}'''


def label_one(example):
    response = requests.post(
        f"{base_url}/chat/completions",
        headers={
            "Authorization": f"Bearer {os.environ['TEACHER_API_KEY']}",
            "Content-Type": "application/json",
        },
        json={
            "model": model,
            "messages": [{"role": "user", "content": build_prompt(example)}],
            "temperature": 0,
            "max_tokens": 2500,
            "reasoning_effort": "low",
            "response_format": {
                "type": "json_schema",
                "json_schema": {
                    "name": "fallacy_tutor_response",
                    "strict": True,
                    "schema": BEHAVIOR_SCHEMA,
                },
            },
        },
        timeout=180,
    )
    if response.status_code != 200:
        return {
            "example_id": example["example_id"],
            "valid": False,
            "errors": [f"http_{response.status_code}"],
        }

    body = response.json()
    content = body["choices"][0]["message"].get("content", "")
    try:
        output = json.loads(content)
        missing = set(BEHAVIOR_SCHEMA["required"]) - set(output)
        errors = [f"missing_{field}" for field in sorted(missing)]
    except Exception as exc:
        output = None
        errors = [f"parse_error: {exc}"]

    return {
        "example_id": example["example_id"],
        "valid": not errors,
        "errors": errors,
        "output": output,
        "usage": body.get("usage", {}),
    }


def label_with_retry(example, attempts=3):
    result = None
    for attempt in range(1, attempts + 1):
        try:
            result = label_one(example)
            result["attempt"] = attempt
            if result["valid"]:
                return result
        except Exception as exc:
            result = {
                "example_id": example["example_id"],
                "valid": False,
                "errors": [f"{type(exc).__name__}: {exc}"],
                "attempt": attempt,
            }
        if attempt < attempts:
            time.sleep((2 ** (attempt - 1)) + random.random())
    return result

print("Teacher function ready.")


In [ ]:
PRODUCTION_OUTPUT = Path("/content/drive/MyDrive/cogni/direct_gemini_15k.jsonl")
valid_ids = set()

if PRODUCTION_OUTPUT.exists():
    for line in PRODUCTION_OUTPUT.read_text().splitlines():
        if line.strip():
            previous = json.loads(line)
            if previous.get("valid"):
                valid_ids.add(previous["example_id"])

pending = [
    example for example in all_inputs if example["example_id"] not in valid_ids
]
print(f"Already valid: {len(valid_ids):,}")
print(f"Pending: {len(pending):,}")

started = time.time()
processed = 0
new_valid = 0

with PRODUCTION_OUTPUT.open("a", encoding="utf-8") as output_file:
    with ThreadPoolExecutor(max_workers=32) as executor:
        futures = [executor.submit(label_with_retry, example) for example in pending]
        for future in as_completed(futures):
            result = future.result()
            output_file.write(json.dumps(result, ensure_ascii=False) + "\n")
            output_file.flush()
            processed += 1
            new_valid += int(result["valid"])

            if processed % 100 == 0:
                elapsed = time.time() - started
                rate = processed / elapsed
                remaining = len(pending) - processed
                eta = remaining / rate / 3600 if rate else 0
                print(
                    f"Processed {processed:,}/{len(pending):,} | "
                    f"Valid {new_valid:,} | Success {new_valid / processed:.1%} | "
                    f"ETA {eta:.1f} hours"
                )

print("Labeling complete.")


In [ ]:
latest_valid = {}
for line in PRODUCTION_OUTPUT.read_text().splitlines():
    if line.strip():
        row = json.loads(line)
        if row.get("valid") and isinstance(row.get("output"), dict):
            latest_valid[row["example_id"]] = row["output"]

inputs_by_id = {example["example_id"]: example for example in all_inputs}
SYSTEM_PROMPT = (
    "You are an AP English Language logical-fallacy tutor. "
    "Follow the seven-section instructional sequence exactly."
)


def render_answer(output):
    assumptions = "\n".join(f"- {value}" for value in output["assumptions"])
    analogy = output["cross_domain_analogy"]
    transfer = output["transfer_check"]
    return f'''## Argument Summary
{output["argument_summary"]}

## Assumptions
{assumptions}

## Primary Fallacy
{output["primary_fallacy"]}

## Why This Applies
{output["why_this_applies"]}

## Cross-Domain Analogy
{analogy["analogy"]}

Mapping: {analogy["mapping"]}

## Transfer Check
{transfer["example"]}

{transfer["question"]}

## Reflective Question
{output["reflective_question"]}'''


compiled = {"train": [], "validation": [], "test": []}
for example_id, output in latest_valid.items():
    source = inputs_by_id[example_id]
    compiled[source["split"]].append(
        {
            "id": example_id,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": source["essay"]},
                {"role": "assistant", "content": render_answer(output)},
            ],
            "metadata": {"source": source["source"]},
        }
    )

DATA_DIR = Path("/content/drive/MyDrive/cogni/production_v1")
DATA_DIR.mkdir(parents=True, exist_ok=True)
for split, rows in compiled.items():
    with (DATA_DIR / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")

counts = {split: len(rows) for split, rows in compiled.items()}
print(counts)
print("Total:", sum(counts.values()))
assert sum(counts.values()) >= 10_000
assert counts["train"] >= 9_000
print("Dataset ready for training.")


In [ ]:
archive = shutil.make_archive(
    "/content/drive/MyDrive/cogni/production_v1", "zip", DATA_DIR
)
print("Dataset folder:", DATA_DIR)
print("ZIP:", archive)


After the final cell succeeds, open `train_ap_tutor_production_colab.ipynb`, select an
A100 GPU runtime, and run the training notebook. Do not use a GPU for this labeling notebook.
